# Temas 2 - pandas

## Bloque 1. que es un DataFrame

Es una tabla: filas y columnas con nombre, como una hoja de Excel o una tabla de SQL.

Cada Columna es una Serie -> una lista de valores con nombre y todos del mismo tipo. 

                  ┌────────── DataFrame (la tabla entera) ──────────┐
         índice   station      date            value    quality
           0      Stockholm    1730865600000   -1.4     G
           1      Stockholm    1730869200000   -2.0     G
           2      Malmö        1730865600000    2.1     Y
                               └── Series ──┘  (una columna)
        

Antes trabajamos en el Tema 1 trabajaste con listas de diccionarios, y cada operacion necesitaba un for:
- filtrar por calidad, convertir valores, calcular minimos.

Pandas trabaja con la tabla completa de una vez. Lo que llevo 15 lineas en Repaso 5, en pandas son 3 o 4. 

Es la herramienta estandar para analizar datos en Python, y las API de los DataFrames de Spark se le parece mucho. 

In [2]:
import pandas as pd   # convención universal: siempre "pd"

mediciones = [
    {"station": "Stockholm", "date": 1730865600000, "value": "-1.4", "quality": "G"},
    {"station": "Stockholm", "date": 1730869200000, "value": "-2.0", "quality": "G"},
    {"station": "Malmö",     "date": 1730865600000, "value": "2.1",  "quality": "Y"},
]

df = pd.DataFrame(mediciones)   # lista de dicts → tabla: cada dict es una fila, cada clave una columna
df                              # en un notebook, la última línea de una celda se muestra sola, como tabla

,station,date,value,quality
0,Stockholm,1730865600000,-1.4,G
1,Stockholm,1730869200000,-2.0,G
2,Malmö,1730865600000,2.1,Y


In [3]:
print(type(df))          # <class 'pandas.core.frame.DataFrame'>
print(type(df["value"])) # <class 'pandas.core.series.Series'> → una columna
print(df.shape)          # (3, 4) → (filas, columnas)
print(df.columns)        # los nombres de las columnas
print(df.dtypes)         # el tipo de cada columna

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
(3, 4)
Index(['station', 'date', 'value', 'quality'], dtype='object')
station    object
date        int64
value      object
quality    object
dtype: object


### Practica guiada - Paso 1

In [4]:
respuesta_malmo = {
    "station": {"key": "52350", "name": "Malmö"},
    "value": [
        {"date": 1730887200000, "value": "2.1", "quality": "G"},
        {"date": 1730890800000, "value": "2.6", "quality": "Y"},
        {"date": 1730894400000, "value": "3.0", "quality": "G"},
    ],
}

In [5]:
values_malmo = respuesta_malmo['value']
df_malmo = pd.DataFrame(values_malmo)
print(df_malmo.shape)
print(df_malmo.columns)        # los nombres de las columnas
print(df_malmo.dtypes)
df_malmo

(3, 3)
Index(['date', 'value', 'quality'], dtype='object')
date        int64
value      object
quality    object
dtype: object


,date,value,quality
0,1730887200000,2.1,G
1,1730890800000,2.6,Y
2,1730894400000,3.0,G


### Practica guiada - Paso: la otra forma de crear un DF

In [6]:
datos = {
    "ciudad": ["Visby", "Ystad", "Kiruna", "Lund"],
    "temp": [5.1, 6.0, -12.4, 6.3],
}
df_ciudades = pd.DataFrame(datos)
print(df_ciudades)
print(df_ciudades.shape)
print(df_ciudades.dtypes)
df_ciudades["pais"] = "SE"
print(df_ciudades.shape)
df_ciudades

   ciudad  temp
0   Visby   5.1
1   Ystad   6.0
2  Kiruna -12.4
3    Lund   6.3
(4, 2)
ciudad     object
temp      float64
dtype: object
(4, 3)


,ciudad,temp,pais
0,Visby,5.1,SE
1,Ystad,6.0,SE
2,Kiruna,-12.4,SE
3,Lund,6.3,SE


In [7]:
lecturas = [
    {"ciudad": "Visby", "temp": 5.1},
    {"ciudad": "Ystad"},
    {"ciudad": "Kiruna", "temp": -12.4, "viento": 7.2},
]
df_lecturas = pd.DataFrame(lecturas)
df_lecturas

,ciudad,temp,viento
0,Visby,5.1,NaN
1,Ystad,NaN,NaN
2,Kiruna,-12.4,7.2


## Bloque 2: leer y exportar datos

**pd.read_csv()** -> lee un archivo CSV y lo convierte en un DataFrame.

El primer paso con cualquier dato nuevo es explorarlo:

- cuantas filas tiene, que tipos tienen las columnas, si hay nullos y si hay valores raros. 

Saltarse esto es el origen de la mayoria de los bugs en pipelines. En Databricks haras lo mismo con **spark.read.csv** y **display()**

Un problema sueco -> los CSV suecos u muchos europeos usan ; como separador y , como decimal. 

Pandas por defecto espera lo contrario: , como reparador y . como decimal.

In [8]:
df = pd.read_csv("datos/observaciones.csv", sep=";", decimal=",")
df.head()
df.shape     # las 5 primeras filas (df.head(10) para 10)

(10, 5)

**Parametro** sep=";" -> que caracter separa las columnas
**Parametro** decimal="." -> que caracter separa los decimales

Las funciones para explorar:

**df.head() / df.tail()**	Las primeras / últimas filas	Ver qué pinta tienen los datos

**df.shape**	(filas, columnas)	Saber el tamaño

**df.dtypes**	El tipo de cada columna	Comprobar que los números son números

**df.info()**	Tipos + cuántos valores no nulos tiene cada columna	Detectar nulos de un vistazo

**df.describe()**	Estadísticas de las columnas numéricas: count, mean, std, min, cuartiles, max	Detectar valores raros (un 99 °C, por ejemplo)

### Practica guiada - Paso 3: el error tipico primero

In [9]:
df_mal = pd.read_csv("datos/observaciones.csv")
df_mal.head()

,station;datum;tid;temperatur;kvalitet
Stockholm;2024-11-06;05:00;-1,4;G
Stockholm;2024-11-06;06:00;-2,0;G
Stockholm;2024-11-06;07:00;;G,NaN
Malmö;2024-11-06;05:00;2,1;Y
Malmö;2024-11-06;06:00;2,6;G


In [10]:
df_mal.shape

(10, 1)

### Practica guiada - Paso 4: leer bien y explorar

1. Lee el archivo correctamente, con sep y decimal, en una variable df.
2. Antes de ejecutar las siguientes líneas, predice:
    - ¿Qué shape tendrá ahora?
    - ¿Qué dtype tendrá temperatur: object o float64? ¿Por qué?
    - En df.info(), ¿cuántos valores no nulos tendrá temperatur? Mira bien el CSV.
3. Ejecuta df.shape, df.info() y df.describe(), cada uno en su celda.

In [11]:
df = pd.read_csv("datos/observaciones.csv", sep=";", decimal=",")
df.head()

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G


In [12]:
df.shape

(10, 5)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   station     10 non-null     object 
 1   datum       10 non-null     object 
 2   tid         10 non-null     object 
 3   temperatur  9 non-null      float64
 4   kvalitet    10 non-null     object 
dtypes: float64(1), object(4)
memory usage: 532.0+ bytes


In [14]:
df.describe()

,temperatur
count,9.000000
mean,-5.222222
std,7.407391
min,-13.100000
25%,-12.700000
50%,-2.000000
75%,2.100000
max,3.000000


### Bloque 3: seleccionar y filtrar

Quedarte solo con algunas columnas (como el SELECT de SQL) o con algunas filas que cumplen una condicion (como el WHERE).

Casi todo el analisis empieza asi:
- "Solo las mediciones de calidad G", "solo la temperatura y la hora"


In [15]:
df["temperatur"]                  # UNA columna → Series

0    -1.4
1    -2.0
2     NaN
3     2.1
4     2.6
5     3.0
6   -12.4
7   -13.1
8   -13.1
9   -12.7
Name: temperatur, dtype: float64

In [16]:
df[["station", "temperatur"]]     # VARIAS columnas → DataFrame (fíjate en los DOBLES corchetes)

,station,temperatur
0,Stockholm,-1.4
1,Stockholm,-2.0
2,Stockholm,NaN
3,Malmö,2.1
4,Malmö,2.6
5,Malmö,3.0
6,Kiruna,-12.4
7,Kiruna,-13.1
8,Kiruna,-13.1
9,Kiruna,-12.7


**Filtrat filas:** la mascara booleana

Filtrar en pandas se hace en dos pasos:

In [17]:
# 1. Una Series de TRUE/FALSE: una por fila

mascara = df['kvalitet'] == "G"
mascara

0     True
1     True
2     True
3    False
4     True
5     True
6     True
7     True
8     True
9     True
Name: kvalitet, dtype: bool

In [18]:
# 2. Solo las filas donde la mascara es TRUE
df[mascara]

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [19]:
# lo mismo , en una sola linea (la forma mas habitual)
df[df['kvalitet'] == "G"]

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


**Combinar condiciones**

In [20]:
df[(df['station'] == 'Kiruna') & (df['kvalitet'] == 'G')]

,station,datum,tid,temperatur,kvalitet
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [21]:
df[(df['temperatur'] < -10) | (df['temperatur'] > 2)] # -> or

,station,datum,tid,temperatur,kvalitet
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [22]:
df[~(df['kvalitet'] == 'G')] #NO (lo conrario)

,station,datum,tid,temperatur,kvalitet
3,Malmö,2024-11-06,05:00,2.1,Y


In [23]:
df[df['station'].isin(['Malmö', 'Kiruna'])] # esta en esta lista

,station,datum,tid,temperatur,kvalitet
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


**loc**: filas y columnas a la vez

In [24]:
df.loc[df["station"] == "Kiruna", ["tid", "temperatur"]]
#      └──── qué filas ────┘   └──── qué columnas ────┘
# que filas tienen de nombre estacion Kiruna
# las columnas tid y temperatur de las filas que hemos seleccionado antes

,tid,temperatur
6,05:00,-12.4
7,06:00,-13.1
8,06:00,-13.1
9,07:00,-12.7


### Practica guiada - Paso 5

Usando tu df:

1. Predice sin ejecutar cuántas filas devuelve cada línea:

In [25]:
# df[df["kvalitet"] == "G"]                                    # a)
# df[df["station"] == "Kiruna"]                                # b)
# df[(df["station"] == "Kiruna") & (df["temperatur"] < -13)]   # c)
# df[df["temperatur"] > 0]                                     # d)

a -> todas las filas que tengas kvalitet G -> 9 filas
b -> todas las filas que tengas de vaor Kiruna en la columna station -> 4
c -> todas las filas que tengan de valor Kiruna en la columna station, y tambien solo las filas donde el valor de temperatura se menos de 13 -> 2 filas
d -> todas las filas que tengas de valor en temperatur mas de 0 -> 3 filas

In [26]:
df.loc[(df['station'] == 'Malmö') & (df["kvalitet"] == 'G'), ['tid' , 'temperatur']]

,tid,temperatur
4,06:00,2.6
5,07:00,3.0


## Bloque 4a: nulos y duplicados

Es lo que se hace de pasar a bronze a silver en el MODULO3:tratarnulos,quitar duplicados , estandadrizar tipos.

In [27]:
df.isna() # tabla de TRUE/FALSE: True donde hay NaN


,station,datum,tid,temperatur,kvalitet
0,False,False,False,False,False
1,False,False,False,False,False
2,False,False,False,True,False
3,False,False,False,False,False
4,False,False,False,False,False
5,False,False,False,False,False
6,False,False,False,False,False
7,False,False,False,False,False
8,False,False,False,False,False
9,False,False,False,False,False


In [28]:
df.isna().sum() # cuantos NaN hay en cada columna

station       0
datum         0
tid           0
temperatur    1
kvalitet      0
dtype: int64

In [29]:
df[df['temperatur'].isna()] # las filas con temperatura nula

,station,datum,tid,temperatur,kvalitet
2,Stockholm,2024-11-06,07:00,NaN,G


In [30]:
df[df['temperatur'].notna()]

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


**Hay dos formas de tratar nulos:**

df.dropna(subset=['temperatur'])

- Elimina las filas con NaN en esa columna
  
- Se usa cuando el dato es imprescindible y no tiene sentido invertarlo

df['temperatur'].fillna(valor)

- Rellena los NaN con un valor

- Hay un valor razonable de reemplazo(la media, la mediana, 0...)

**subset** dice en que columnas mirar. Sin **subset, dropna()** elimina cualquier fila que tenga algun NaN, en cualquier columna, y puedes perder filas sin darte cuenta. 


**Duplicados**


In [31]:
df.duplicated() # True en las filas que repiten una anterios

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8     True
9    False
dtype: bool

In [32]:
df.duplicated().sum() # cuants hay

np.int64(1)

In [33]:
df.drop_duplicates() # sin las filas repetidas (identicas en TODO)

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [ ]:
df.drop_duplicates(subset=['station', 'datum', 'tid'])
#repetidas solo en estas columnas

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Stockholm,2024-11-06,07:00,NaN,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


**subset** en duplicados es clave. Si una estacion envia dos mediciones para la misma hora, aunque sean distintas, es un duplicado: una estacion no mide dos veces a la misma hora. Por eso se define que columnas identifican una medicion unica que en nuestro caso son : estacion + fecha + hora

**La trampa mas importante de pandas**

Estos metodos no modifican el DF original. Devuelve uno nuevo:

In [ ]:
df.dropna(subset=['temperatur'])
# crea una tabla nueva... y despues se pierde si no se guarda


,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
3,Malmö,2024-11-06,05:00,2.1,Y
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
8,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [36]:
print(df.shape)

(10, 5)


In [37]:
df_limpio = df.dropna(subset=['temperatur'])

### Practica guiada - Paso 6

1. Predice sin ejecutar
   - ¿Qué devuelve df.isna().sum() para cada una de las 5 columnas?
   - ¿Qué devuelve df.duplicated().sum()?

devolvera 1 ya que solo hay un NaN en todo el dataset
devolvera 1 por que hay un duplicado que es la file 8

2. Crea df_limpio a partir de df, encadenando estas operaciones, cada una guardada en df_limpio:
   - Quitar las filas con temperatur nula.
   - Quitar los duplicados según estación + fecha + hora.
   - Quedarte solo con calidad "G".

In [38]:
df_limpio = df_limpio.drop_duplicates(subset=['station', 'datum', 'tid'])

In [39]:
df_limpio = df_limpio[df_limpio['kvalitet'] == 'G']

In [40]:
df_limpio

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
4,Malmö,2024-11-06,06:00,2.6,G
5,Malmö,2024-11-06,07:00,3.0,G
6,Kiruna,2024-11-06,05:00,-12.4,G
7,Kiruna,2024-11-06,06:00,-13.1,G
9,Kiruna,2024-11-06,07:00,-12.7,G


In [41]:
df.shape

(10, 5)

In [42]:
df_limpio.shape

(7, 5)

In [43]:
df_limpio = df_limpio.reset_index(drop=True)
df_limpio

,station,datum,tid,temperatur,kvalitet
0,Stockholm,2024-11-06,05:00,-1.4,G
1,Stockholm,2024-11-06,06:00,-2.0,G
2,Malmö,2024-11-06,06:00,2.6,G
3,Malmö,2024-11-06,07:00,3.0,G
4,Kiruna,2024-11-06,05:00,-12.4,G
5,Kiruna,2024-11-06,06:00,-13.1,G
6,Kiruna,2024-11-06,07:00,-12.7,G


## Bloque 4b : tipos y fechas

**Convertir tipos**

In [44]:
df['temperatur'].astype(float)
# convierte; Falla si hay un valor imposible de convertir

0    -1.4
1    -2.0
2     NaN
3     2.1
4     2.6
5     3.0
6   -12.4
7   -13.1
8   -13.1
9   -12.7
Name: temperatur, dtype: float64

In [45]:
pd.to_numeric(df_malmo["value"], errors="coerce")
# convierte; lo que no se pueda -> NaN, sin fallar

0    2.1
1    2.6
2    3.0
Name: value, dtype: float64

**Fechas: texto -> datetime**

Tus columnas datum y tid son texto. Para calcular con ellas (ordenar, extraer la hora, convertir zonas), hay que convertirlas

In [ ]:
df_limpio['fecha'] = pd.to_datetime(
    # une los dos textos "2024-11-06 05:00"
    df_limpio['datum'] + " " + df_limpio['tid'],
    # los mismos codigos que strptime
    format="%Y-%m-%d %H:%M",
)
df_limpio

,station,datum,tid,temperatur,kvalitet,fecha
0,Stockholm,2024-11-06,05:00,-1.4,G,2024-11-06 05:00:00
1,Stockholm,2024-11-06,06:00,-2.0,G,2024-11-06 06:00:00
2,Malmö,2024-11-06,06:00,2.6,G,2024-11-06 06:00:00
3,Malmö,2024-11-06,07:00,3.0,G,2024-11-06 07:00:00
4,Kiruna,2024-11-06,05:00,-12.4,G,2024-11-06 05:00:00
5,Kiruna,2024-11-06,06:00,-13.1,G,2024-11-06 06:00:00
6,Kiruna,2024-11-06,07:00,-12.7,G,2024-11-06 07:00:00


**Fechas: epoch en ms** -> datetime UTC (lo de SMHI)

In [47]:
pd.to_datetime(df_malmo['date'], unit='ms', utc=True)

0   2024-11-06 10:00:00+00:00
1   2024-11-06 11:00:00+00:00
2   2024-11-06 12:00:00+00:00
Name: date, dtype: datetime64[ns, UTC]

**El accesor .dt**

Para usar funciones de fecha sobre una columna entera, se pone .dt delante

In [48]:
df_limpio['fecha'].dt.hour # la hora de cada fila

0    5
1    6
2    6
3    7
4    5
5    6
6    7
Name: fecha, dtype: int32

In [49]:
df_limpio["fecha"].dt.strftime("%H:%M")   # texto, como en el Tema 1

0    05:00
1    06:00
2    06:00
3    07:00
4    05:00
5    06:00
6    07:00
Name: fecha, dtype: object

In [50]:
df_limpio["fecha"].dt.day_name() #'Wednesday'...

0    Wednesday
1    Wednesday
2    Wednesday
3    Wednesday
4    Wednesday
5    Wednesday
6    Wednesday
Name: fecha, dtype: object

### Practica guiada - Paso 7

**Parte A** , con df_limpio:
1. Crea la columna fecha uniendo datum y tid, convertida a datetime.
2. Las horas del CSV son hora sueca. Crea fecha_utc con esas fechas convertidas a UTC. Piensa: ¿qué método va primero, tz_localize o tz_convert? ¿Hacen falta los dos?
3. Predice qué hora UTC tendrá la fila de Stockholm 05:00 (recuerda: en noviembre, Suecia está en UTC+1).

In [51]:
df_limpio

,station,datum,tid,temperatur,kvalitet,fecha
0,Stockholm,2024-11-06,05:00,-1.4,G,2024-11-06 05:00:00
1,Stockholm,2024-11-06,06:00,-2.0,G,2024-11-06 06:00:00
2,Malmö,2024-11-06,06:00,2.6,G,2024-11-06 06:00:00
3,Malmö,2024-11-06,07:00,3.0,G,2024-11-06 07:00:00
4,Kiruna,2024-11-06,05:00,-12.4,G,2024-11-06 05:00:00
5,Kiruna,2024-11-06,06:00,-13.1,G,2024-11-06 06:00:00
6,Kiruna,2024-11-06,07:00,-12.7,G,2024-11-06 07:00:00


In [52]:
fechas_utc = df_limpio['fecha']
fechas_utc = fechas_utc.dt.tz_localize('Europe/Stockholm')

In [53]:
df_limpio["fecha_utc"] = df_limpio["fecha"].dt.tz_localize("Europe/Stockholm").dt.tz_convert("UTC")

In [54]:
df_limpio[["station", "fecha", "fecha_utc"]]

,station,fecha,fecha_utc
0,Stockholm,2024-11-06 05:00:00,2024-11-06 04:00:00+00:00
1,Stockholm,2024-11-06 06:00:00,2024-11-06 05:00:00+00:00
2,Malmö,2024-11-06 06:00:00,2024-11-06 05:00:00+00:00
3,Malmö,2024-11-06 07:00:00,2024-11-06 06:00:00+00:00
4,Kiruna,2024-11-06 05:00:00,2024-11-06 04:00:00+00:00
5,Kiruna,2024-11-06 06:00:00,2024-11-06 05:00:00+00:00
6,Kiruna,2024-11-06 07:00:00,2024-11-06 06:00:00+00:00


**Parte B** epoch de SMHI → hora sueca, con df_malmo

Trabajas con df_malmo del Paso 1, que tiene la columna date con los epochs en milisegundos.

1. Predice primero, sin ejecutar, las tres horas en formato HH:MM, hora sueca. Pista: son los mismos epochs del Paso 7 del Tema 1, que en UTC eran 10:00, 11:00 y 12:00.
2. Crea en df_malmo una columna nueva llamada hora_suecia, encadenando tres eslabones, sin for:
    - Eslabón 1: epoch en ms → datetime en UTC, con pd.to_datetime(...) y los parámetros de la unidad y de UTC.
    - Eslabón 2: UTC → hora sueca (aquí no hace falta tz_localize, porque ya tiene la etiqueta UTC).
    - Eslabón 3: fecha → texto HH:MM, con el método de .dt que da formato.
3. Muestra df_malmo con su nueva columna y compara con tu predicción.

In [55]:
df_malmo

,date,value,quality
0,1730887200000,2.1,G
1,1730890800000,2.6,Y
2,1730894400000,3.0,G


In [68]:
df_malmo['fecha'] = pd.to_datetime(df_malmo["date"], unit="ms", utc=True).dt.tz_convert('Europe/Stockholm').dt.strftime("%H:%M")

In [69]:
df_malmo

,date,value,quality,fecha,fecha_utc
0,1730887200000,2.1,G,11:00,2024-11-06 10:00:00+00:00
1,1730890800000,2.6,Y,12:00,2024-11-06 11:00:00+00:00
2,1730894400000,3.0,G,13:00,2024-11-06 12:00:00+00:00


In [ ]:
# df_malmo["fecha_utc"] = pd.to_datetime(df_malmo["date"], unit="ms", utc=True)

# df_malmo["hora_suecia"] = (
#     df_malmo["fecha_utc"]
#     .dt.tz_convert("Europe/Stockholm")
#     .dt.strftime("%H:%M")
# )